# Support Ticket Env - GRPO Fine-Tuning
**OpenEnv x Scalar Hackathon**

Fine-tunes `Qwen/Qwen2.5-0.5B-Instruct` using GRPO + LoRA (PEFT) against the live Support Ticket Environment API.

- Model: Qwen2.5-0.5B-Instruct
- Algorithm: GRPO
- Environment: https://algocore-support-ticket-env.hf.space
- Runtime: ~30-45 min on Kaggle P100/T4 (or Colab)
- Note: Uses standard HuggingFace transformers + PEFT (no Unsloth)

In [ ]:
# Install dependencies (no Unsloth)
!pip install -q 'trl>=0.18.2,<=0.24.0' 'transformers>=4.51.3,<=5.5.0' 'datasets>=3.4.1,<4.4.0' accelerate peft
!pip install -q bitsandbytes requests matplotlib
print('Installation complete')

In [ ]:
import os

# Try Colab secrets first, then Kaggle secrets, then env var
HF_TOKEN = ''

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN') or ''
except Exception:
    pass

if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN') or ''
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

if not HF_TOKEN:
    raise ValueError('HF_TOKEN not found. On Kaggle: Add-ons -> Secrets -> Add new secret (label=HF_TOKEN). On Colab: key icon -> Secrets.')

print('HF_TOKEN loaded OK')

ENV_BASE_URL = 'https://algocore-support-ticket-env.hf.space'
MODEL_NAME   = 'Qwen/Qwen2.5-0.5B-Instruct'
# Auto-detect runtime
RUNTIME     = 'kaggle' if os.path.exists('/kaggle/working') else 'colab'
OUTPUT_DIR  = '/kaggle/working/support-ticket-grpo' if RUNTIME == 'kaggle' else '/content/support-ticket-grpo'
RESULTS_IMG = '/kaggle/working/grpo_results.png' if RUNTIME == 'kaggle' else '/content/grpo_results.png'
HF_REPO_ID  = 'AlgoCore/support-ticket-grpo-model'
print(f'Runtime: {RUNTIME} | Output: {OUTPUT_DIR}')

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - switch runtime!')
if torch.cuda.is_available():
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
print('Model:', MODEL_NAME)
print('Env:', ENV_BASE_URL)

In [ ]:
import requests, json, re, random
from dataclasses import dataclass
from typing import Optional

TICKETS = [
    {'id':'T001','text':'I was charged twice for my subscription this month.','category':'billing','correct_action':'reply'},
    {'id':'T002','text':'I cannot log into my account. Password reset email never arrives.','category':'account','correct_action':'reply'},
    {'id':'T003','text':'Your app crashes every time I upload a file larger than 10 MB.','category':'technical','correct_action':'escalate'},
    {'id':'T004','text':'I want a full refund. I have not used the service at all.','category':'refund','correct_action':'reply'},
    {'id':'T005','text':'What are your business hours and do you have a phone number?','category':'general','correct_action':'reply'},
    {'id':'T006','text':'My invoice shows a charge for a plan I never subscribed to.','category':'billing','correct_action':'escalate'},
    {'id':'T007','text':'How do I cancel my subscription? I cannot find the option.','category':'account','correct_action':'reply'},
    {'id':'T008','text':'The API is returning 500 errors intermittently for 2 hours.','category':'technical','correct_action':'escalate'},
    {'id':'T009','text':'Thank you! The issue has been resolved. You guys are awesome.','category':'general','correct_action':'close'},
    {'id':'T010','text':'I need an itemised invoice for my company accounting department.','category':'billing','correct_action':'reply'},
]

KEYWORD_REWARDS = {
    'billing':   ['charge','invoice','payment','billing','refund'],
    'account':   ['password','login','account','cancel','subscription'],
    'technical': ['engineering','escalate','bug','crash','error'],
    'refund':    ['refund','return','credit','process'],
    'general':   ['hours','contact','phone','information','help'],
}

@dataclass
class Obs:
    ticket_id: str
    ticket_text: str
    task_id: int
    current_category: Optional[str]
    resolved: bool
    step_count: int
    feedback: str
    score: float
    reward: float
    done: bool

class LocalEnv:
    """Mirrors the live HF Space environment exactly. Used as fallback when API is unreachable."""
    def reset(self, task_id=1, seed=42):
        rng = random.Random(seed)
        self.task_id = task_id
        self.ticket  = rng.choice(TICKETS)
        self.classified = False
        self.step_count = 0
        return Obs(self.ticket['id'], self.ticket['text'], task_id,
                   None, False, 0, 'New ticket. Take action.', 0.0, 0.0, False)
    def step(self, action):
        self.step_count += 1
        at    = action.get('action_type', '')
        cat   = action.get('category', '')
        reply = action.get('reply_text', '')
        reward = 0.0; done = False
        if self.task_id == 1:
            reward = 1.0 if cat == self.ticket['category'] else 0.0
            done = True
        elif self.task_id == 2:
            if not self.classified:
                reward = 0.3 if cat == self.ticket['category'] else 0.1
                self.classified = True
            else:
                reward = 1.0 if at == self.ticket['correct_action'] else 0.0
                done = True
        else:
            if not self.classified:
                reward = 0.2 if cat == self.ticket['category'] else 0.0
                self.classified = True
            else:
                action_score = 0.4 if at == self.ticket['correct_action'] else 0.0
                kws = KEYWORD_REWARDS.get(self.ticket['category'], [])
                reply_score = min(0.25, sum(0.05 for kw in kws if kw in reply.lower()))
                reward = action_score + reply_score
                done = True
        return Obs(self.ticket['id'], self.ticket['text'], self.task_id,
                   self.ticket['category'] if self.classified else None,
                   done, self.step_count, f'reward={reward:.2f}', reward, reward, done)

class RemoteEnv:
    """Connects to the live HF Space API at algocore-support-ticket-env.hf.space"""
    def __init__(self, base_url):
        self.base_url = base_url.rstrip('/')
        self.session = requests.Session()
        self.session.headers.update({'Content-Type': 'application/json'})
    def health(self):
        try:
            r = self.session.get(f'{self.base_url}/health', timeout=8)
            return r.status_code == 200
        except: return False
    def reset(self, task_id=1, seed=42):
        r = self.session.post(f'{self.base_url}/reset', json={'task_id': task_id, 'seed': seed}, timeout=15)
        r.raise_for_status()
        obs = r.json().get('observation', r.json())
        return Obs(**{k: obs.get(k, v) for k, v in Obs.__dataclass_fields__.items()})
    def step(self, action):
        r = self.session.post(f'{self.base_url}/step', json={'action': action}, timeout=15)
        r.raise_for_status()
        obs = r.json().get('observation', r.json())
        return Obs(**{k: obs.get(k, v) for k, v in Obs.__dataclass_fields__.items()})

# Auto-select: use live API if reachable, otherwise use local mirror
_remote = RemoteEnv(ENV_BASE_URL)
if _remote.health():
    env_client = _remote
    print('Using LIVE environment:', ENV_BASE_URL)
else:
    env_client = LocalEnv()
    print('Live API unreachable - using LOCAL environment mirror (same reward logic)')

obs = env_client.reset(task_id=1, seed=42)
print(f'Ticket: {obs.ticket_id} - {obs.ticket_text[:60]}')


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

MAX_SEQ_LENGTH = 1024
print(f'Loading {MODEL_NAME} with standard HuggingFace + PEFT...')

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

# 4-bit quantization config via bitsandbytes
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    token=HF_TOKEN,
)

# Prepare model for k-bit training (cast norms to float32 for stability)
# NOTE: do NOT enable gradient_checkpointing here - we toggle it manually per phase
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)

# Apply LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
)
model = get_peft_model(model, lora_config)

print('Model loaded with standard 4-bit LoRA (no Unsloth)')
model.print_trainable_parameters()

In [ ]:
SYSTEM_PROMPT = '''You are a customer support AI agent. Given a ticket, respond with a JSON action.

Respond ONLY with valid JSON (no markdown):
{"action_type": "classify"|"reply"|"escalate"|"close", "category": "billing"|"technical"|"account"|"general"|"refund", "reply_text": "...", "reason": "..."}

Rules:
- Task 1: action_type=classify, pick correct category
- Task 2: first classify, then reply/escalate/close
- Task 3: classify each ticket then resolve it
- category only needed for classify
- reply_text only needed for reply
- technical issues -> escalate
- resolved/thank you -> close
- billing/account/refund/general -> reply'''

def build_prompt(obs):
    user_msg = json.dumps({
        'ticket_id': obs.ticket_id,
        'ticket_text': obs.ticket_text,
        'task_id': obs.task_id,
        'current_category': obs.current_category,
        'feedback': obs.feedback,
        'step_count': obs.step_count,
    }, indent=2)
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_msg},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def parse_action(text):
    text = text.strip()
    text = re.sub(r'^```(?:json)?\s*', '', text)
    text = re.sub(r'\s*```$', '', text)
    try: return json.loads(text)
    except:
        match = re.search(r'\{.*?\}', text, re.DOTALL)
        if match:
            try: return json.loads(match.group())
            except: pass
    return {'action_type': 'classify', 'category': 'general'}

print('Prompt builder OK')

In [ ]:
import random

SEEDS     = [42, 7, 123, 0, 99]
TASK_IDS  = [1, 2, 3]
MAX_STEPS = 6

def generate_action(prompt_text, max_new_tokens=150):
    # Disable gradient checkpointing for inference so use_cache works
    model.eval()
    model.config.use_cache = True
    inputs = tokenizer(prompt_text, return_tensors='pt', truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    decoded = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return decoded

def run_episode(task_id, seed, verbose=False):
    obs = env_client.reset(task_id=task_id, seed=seed)
    prompts, completions, rewards = [], [], []
    done = False
    for step in range(MAX_STEPS):
        if done: break
        p = build_prompt(obs)
        c = generate_action(p)
        action = parse_action(c)
        if verbose: print(f'  Step {step+1}: action={action}')
        try:
            obs = env_client.step(action)
            reward = float(obs.reward or 0.0)
            done = obs.done
        except Exception as e:
            print(f'  [step error] {e}')
            reward = 0.0
            done = True
        prompts.append(p); completions.append(c); rewards.append(reward)
        if done: break
    return prompts, completions, sum(rewards)

print('Running smoke test (verbose)...')
p, c, r = run_episode(task_id=1, seed=42, verbose=True)
print(f'Smoke test done - steps={len(p)}, total_reward={r:.3f}')
if len(p) == 0:
    print('WARNING: 0 steps - generation may be broken, check output above')

In [ ]:
def evaluate(n_seeds=3):
    results = {}
    for task_id in [1, 2, 3]:
        task_rewards = []
        for seed in SEEDS[:n_seeds]:
            _, _, total = run_episode(task_id=task_id, seed=seed)
            task_rewards.append(round(max(0, min(1, total / MAX_STEPS)), 3))
        avg = round(sum(task_rewards) / len(task_rewards), 3)
        results[f'task{task_id}'] = avg
        print(f'  Task {task_id}: {avg:.3f}')
    results['overall'] = round(sum(results.values()) / 3, 3)
    print(f'  Overall: {results["overall"]:.3f}')
    return results

print('=== BASELINE (before training) ===')
baseline_scores = evaluate(n_seeds=3)

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
import numpy as np

LEARNING_RATE = 5e-5
N_EPISODES    = 60
GROUP_SIZE    = 4
KL_COEFF      = 0.01
GRAD_CLIP     = 1.0
LOG_EVERY     = 5

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=5, num_training_steps=N_EPISODES)
training_log = []

print(f'Starting GRPO training (standard HF+PEFT): {N_EPISODES} episodes, group_size={GROUP_SIZE}')
print('=' * 60)

for episode in range(1, N_EPISODES + 1):
    task_id = random.choice(TASK_IDS)
    seed    = random.choice(SEEDS)

    group_rewards, group_prompts, group_completions = [], [], []

    # Inference phase
    model.eval()
    for g in range(GROUP_SIZE):
        obs = env_client.reset(task_id=task_id, seed=seed)
        prompt = build_prompt(obs)
        completion = generate_action(prompt)
        action = parse_action(completion)
        try:
            obs = env_client.step(action)
            reward = float(obs.reward or 0.0)
        except:
            reward = -0.1
        group_rewards.append(reward)
        group_prompts.append(prompt)
        group_completions.append(completion)

    rewards_arr = np.array(group_rewards, dtype=np.float32)
    advantages  = (rewards_arr - rewards_arr.mean()) / (rewards_arr.std() + 1e-8)

    # Training phase - re-enable gradient checkpointing
    model.train()
    model.config.use_cache = False
    model.gradient_checkpointing_enable()
    optimizer.zero_grad()
    total_loss = torch.tensor(0.0, device=next(model.parameters()).device)

    for prompt, completion, adv in zip(group_prompts, group_completions, advantages):
        if not completion.strip(): continue
        full_text = prompt + completion
        inputs = tokenizer(full_text, return_tensors='pt', truncation=True, max_length=1200).to(model.device)
        prompt_len = tokenizer(prompt, return_tensors='pt')['input_ids'].shape[1]
        outputs = model(**inputs, labels=inputs['input_ids'])
        logits = outputs.logits[:, prompt_len-1:-1, :]
        target_ids = inputs['input_ids'][:, prompt_len:]
        if target_ids.shape[1] == 0: continue
        log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
        token_log_probs = log_probs.gather(2, target_ids.unsqueeze(-1)).squeeze(-1)
        seq_log_prob = token_log_probs.mean()
        pg_loss  = -torch.tensor(float(adv), device=model.device) * seq_log_prob
        kl_loss  = KL_COEFF * (seq_log_prob ** 2)
        total_loss = total_loss + (pg_loss + kl_loss) / GROUP_SIZE

    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    optimizer.step()
    scheduler.step()

    avg_reward = float(rewards_arr.mean())
    training_log.append((episode, task_id, avg_reward))

    if episode % LOG_EVERY == 0:
        print(f'Episode {episode:3d}/{N_EPISODES} | task={task_id} | avg_reward={avg_reward:.3f} | loss={total_loss.item():.4f}')

print('Training complete!')

In [ ]:
model.eval()

print('=== POST-TRAINING EVALUATION ===')
trained_scores = evaluate(n_seeds=3)

print('\n=== IMPROVEMENT SUMMARY ===')
print(f'{"Task":<10} {"Before":>8} {"After":>8} {"Delta":>8}')
print('-' * 38)
for key, label in [('task1','Task 1'),('task2','Task 2'),('task3','Task 3'),('overall','Overall')]:
    b = baseline_scores.get(key, 0)
    a = trained_scores.get(key, 0)
    print(f'{label:<10} {b:>8.3f} {a:>8.3f} {a-b:>+8.3f}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

episodes_log   = [x[0] for x in training_log]
task_ids_log   = [x[1] for x in training_log]
ep_rewards_log = [x[2] for x in training_log]

def moving_avg(data, window=5):
    return np.convolve(data, np.ones(window)/window, mode='valid')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Support Ticket Env - GRPO Training Results', fontsize=14, fontweight='bold')

ax1 = axes[0]
colors = {1: '#3498db', 2: '#2ecc71', 3: '#e74c3c'}
for tid in [1, 2, 3]:
    mask = [i for i, t in enumerate(task_ids_log) if t == tid]
    if mask:
        x = [episodes_log[i] for i in mask]
        y = [ep_rewards_log[i] for i in mask]
        ax1.scatter(x, y, alpha=0.3, color=colors[tid], s=15)
        if len(y) >= 5:
            ax1.plot(x[2:-2], moving_avg(y), color=colors[tid], linewidth=2, label=f'Task {tid}')
        else:
            ax1.plot(x, y, color=colors[tid], linewidth=2, label=f'Task {tid}')

ax1.set_xlabel('Episode'); ax1.set_ylabel('Avg Reward')
ax1.set_title('Training Reward per Episode')
ax1.legend(); ax1.grid(True, alpha=0.3); ax1.set_ylim(-0.1, 1.1)

ax2 = axes[1]
tasks = ['Task 1', 'Task 2', 'Task 3', 'Overall']
keys  = ['task1', 'task2', 'task3', 'overall']
bv = [baseline_scores.get(k, 0) for k in keys]
av = [trained_scores.get(k, 0) for k in keys]
x = np.arange(len(tasks)); w = 0.35
b1 = ax2.bar(x - w/2, bv, w, label='Before Training', color='#95a5a6')
b2 = ax2.bar(x + w/2, av, w, label='After GRPO', color='#2ecc71')
for bar in b1:
    ax2.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.01, f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)
for bar in b2:
    ax2.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.01, f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold', color='#27ae60')
ax2.set_xticks(x); ax2.set_xticklabels(tasks)
ax2.set_ylabel('Score (0-1)'); ax2.set_title('Before vs After GRPO')
ax2.legend(); ax2.grid(True, alpha=0.3, axis='y'); ax2.set_ylim(0, 1.15)

plt.tight_layout()
plt.savefig(RESULTS_IMG, dpi=150, bbox_inches='tight')
plt.show()
print(f'Chart saved to {RESULTS_IMG}')

In [ ]:
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Model saved to {OUTPUT_DIR}')

try:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(HF_REPO_ID, exist_ok=True, private=False)
    api.upload_folder(folder_path=OUTPUT_DIR, repo_id=HF_REPO_ID, repo_type='model')
    api.upload_file(path_or_fileobj=RESULTS_IMG, path_in_repo='grpo_results.png', repo_id=HF_REPO_ID, repo_type='model')
    print(f'Model pushed to: https://huggingface.co/{HF_REPO_ID}')
except Exception as e:
    print(f'Push failed: {e}')
    print(f'Model saved locally at {OUTPUT_DIR}')

In [ ]:
# Download chart: Kaggle saves to Output tab automatically; Colab needs explicit download
if RUNTIME == 'colab':
    try:
        from google.colab import files
        files.download(RESULTS_IMG)
    except Exception as e:
        print(f'Download skipped: {e}')
else:
    print(f'Kaggle: chart saved to Output tab -> {RESULTS_IMG}')

print('\n' + '='*50)
print('FINAL TRAINING SUMMARY')
print('='*50)
print(f'Model:     {MODEL_NAME}')
print(f'Algorithm: GRPO + LoRA (standard HF + PEFT)')
print(f'Episodes:  {N_EPISODES}')
print(f'Env:       {ENV_BASE_URL}')
print()
print(f'{"Task":<10} {"Before":>8} {"After":>8} {"Delta":>8}')
print('-' * 38)
for key, label in [('task1','Task 1'),('task2','Task 2'),('task3','Task 3'),('overall','Overall')]:
    b = baseline_scores.get(key, 0)
    a = trained_scores.get(key, 0)
    print(f'{label:<10} {b:>8.3f} {a:>8.3f} {a-b:>+8.3f}')
print('='*50)
print(f'Model: https://huggingface.co/{HF_REPO_ID}')